In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("data/processed/cleaned_data.csv", parse_dates=["transaction_date"])

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df[['price_per_unit', 'quantity', 'total_spent']].describe()

In [ ]:
numeric_cols = ['price_per_unit', 'quantity', 'total_spent']
skew_kurt = pd.DataFrame({
    'skewness': df[numeric_cols].skew(),
    'kurtosis': df[numeric_cols].kurt()
})
skew_kurt

In [ ]:
corr_matrix = df[numeric_cols].corr()
corr_matrix

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
r, p = stats.pearsonr(df['price_per_unit'], df['total_spent'])
print(f"Pearson r (price_per_unit vs total_spent): {r:.4f}, p-value: {p:.4e}")

In [ ]:
r2, p2 = stats.pearsonr(df['quantity'], df['total_spent'])
print(f"Pearson r (quantity vs total_spent): {r2:.4f}, p-value: {p2:.4e}")

In [ ]:
for col in numeric_cols:
    stat, p_val = stats.kstest(df[col], 'norm', args=(df[col].mean(), df[col].std()))
    print(f"KS Test [{col}]: stat={stat:.4f}, p={p_val:.4e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    ax.hist(df[col], bins=30, edgecolor='black', color='steelblue')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
category_stats = df.groupby('category')['total_spent'].agg(['mean', 'median', 'std', 'count'])
category_stats.columns = ['mean_spent', 'median_spent', 'std_spent', 'txn_count']
category_stats.sort_values('mean_spent', ascending=False)

In [ ]:
groups = [grp['total_spent'].values for _, grp in df.groupby('category')]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"One-Way ANOVA (total_spent across categories): F={f_stat:.4f}, p={p_anova:.4e}")

In [ ]:
plt.figure(figsize=(12, 5))
df.boxplot(column='total_spent', by='category', figsize=(12, 5), grid=False, rot=30)
plt.suptitle('')
plt.title('Total Spent by Category')
plt.xlabel('Category')
plt.ylabel('Total Spent')
plt.tight_layout()
plt.show()

In [ ]:
pay_groups = [grp['total_spent'].values for _, grp in df.groupby('payment_method')]
f2, p2_anova = stats.f_oneway(*pay_groups)
print(f"One-Way ANOVA (total_spent across payment methods): F={f2:.4f}, p={p2_anova:.4e}")

In [ ]:
loc_groups = [grp['total_spent'].values for _, grp in df.groupby('location')]
f3, p3_anova = stats.f_oneway(*loc_groups)
print(f"One-Way ANOVA (total_spent across locations): F={f3:.4f}, p={p3_anova:.4e}")

In [ ]:
contingency_cat_pay = pd.crosstab(df['category'], df['payment_method'])
chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency_cat_pay)
print(f"Chi-Square (category vs payment_method): chi2={chi2:.4f}, p={p_chi2:.4e}, dof={dof}")

In [ ]:
contingency_cat_loc = pd.crosstab(df['category'], df['location'])
chi2_b, p_b, dof_b, _ = stats.chi2_contingency(contingency_cat_loc)
print(f"Chi-Square (category vs location): chi2={chi2_b:.4f}, p={p_b:.4e}, dof={dof_b}")

In [ ]:
contingency_pay_loc = pd.crosstab(df['payment_method'], df['location'])
chi2_c, p_c, dof_c, _ = stats.chi2_contingency(contingency_pay_loc)
print(f"Chi-Square (payment_method vs location): chi2={chi2_c:.4f}, p={p_c:.4e}, dof={dof_c}")

In [ ]:
online = df[df['location'] == 'Online']['total_spent']
instore = df[df['location'] == 'In-store']['total_spent']
t_stat, p_ttest = stats.ttest_ind(online, instore)
print(f"T-Test (Online vs In-store total_spent): t={t_stat:.4f}, p={p_ttest:.4e}")

In [ ]:
discounted = df[df['discount_applied'] == True]['total_spent']
not_discounted = df[df['discount_applied'] == False]['total_spent']
t_d, p_d = stats.ttest_ind(discounted, not_discounted)
print(f"T-Test (Discounted vs Non-discounted total_spent): t={t_d:.4f}, p={p_d:.4e}")

In [ ]:
n = len(df['total_spent'])
mean_ts = df['total_spent'].mean()
se_ts = stats.sem(df['total_spent'])
ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=mean_ts, scale=se_ts)
print(f"95% CI for total_spent: [{ci_low:.4f}, {ci_high:.4f}]")

In [ ]:
n_q = len(df['quantity'])
mean_q = df['quantity'].mean()
se_q = stats.sem(df['quantity'])
ci_q_low, ci_q_high = stats.t.interval(0.95, df=n_q-1, loc=mean_q, scale=se_q)
print(f"95% CI for quantity: [{ci_q_low:.4f}, {ci_q_high:.4f}]")

In [ ]:
category_revenue = df.groupby('category')['total_spent'].sum().sort_values(ascending=False)
category_revenue

In [ ]:
plt.figure(figsize=(10, 5))
category_revenue.plot(kind='bar', color='mediumseagreen', edgecolor='black')
plt.title('Total Revenue by Category')
plt.xlabel('Category')
plt.ylabel('Total Revenue')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
payment_revenue = df.groupby('payment_method')['total_spent'].sum().sort_values(ascending=False)
payment_revenue

In [ ]:
location_revenue = df.groupby('location')['total_spent'].sum()
location_revenue

In [ ]:
discount_revenue = df.groupby('discount_applied')['total_spent'].agg(['sum', 'mean', 'count'])
discount_revenue

In [ ]:
customer_revenue = df.groupby('customer_id')['total_spent'].sum().sort_values(ascending=False)
customer_revenue.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['price_per_unit'], df['total_spent'], alpha=0.3, color='steelblue', edgecolors='none')
plt.xlabel('Price Per Unit')
plt.ylabel('Total Spent')
plt.title('Price Per Unit vs Total Spent')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['quantity'], df['total_spent'], alpha=0.3, color='tomato', edgecolors='none')
plt.xlabel('Quantity')
plt.ylabel('Total Spent')
plt.title('Quantity vs Total Spent')
plt.tight_layout()
plt.show()